# 03 — Machine Learning canônico

Baseline experimental em nível **cliente-mês**, com XGBoost explicável e avaliação temporal.

O objetivo é demonstrar priorização analítica reproduzível. A base é **sintética**, o target é um **label fraco** e nenhuma saída deve ser interpretada como comprovação de ilícito, validação produtiva ou decisão autônoma.


## Contrato experimental

- Unidade analítica: cliente-mês.
- Treino: julho/2025.
- Calibragem e seleção do threshold: agosto/2025.
- Teste temporal preservado: setembro/2025.
- Outubro/2025 é excluído por ser mês incompleto.
- Label canônico: `weak_label`, derivado das regras determinísticas M01–M12.
- R17 permanece fora do label canônico.
- Não há separação PF/PJ porque a base não sustenta classificação confiável para esse recorte.
- Missing e outliers são preservados como informação.
- Não há imputação, normalização ou remoção automática.
- XGBoost com `random_state=42`.
- SHAP e feature importance são explicabilidade pós-hoc.
- O threshold estatístico não representa homologação operacional.
- Revisão humana é obrigatória.


In [ ]:
from pathlib import Path
import json
import os
import sys
import tempfile

import pandas as pd


def find_repo_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in (
        current,
        *current.parents,
    ):
        if (
            (
                candidate
                / "src"
                / "run_ml.py"
            ).is_file()
            and (
                candidate
                / "outputs"
                / "t1_suspects"
                / "04_client_month_alerts_all.csv"
            ).is_file()
        ):
            return candidate

    raise RuntimeError(
        "Raiz do repositório não encontrada."
    )


ROOT = find_repo_root()

os.chdir(
    ROOT
)

src_path = str(
    ROOT
    / "src"
)

if src_path not in sys.path:
    sys.path.insert(
        0,
        src_path,
    )

from run_ml import (
    ARTIFACT_FILENAMES,
    run_pipeline,
    sha256_file,
)


INPUT_PATH = Path(
    "outputs/t1_suspects/"
    "04_client_month_alerts_all.csv"
)

VERSIONED_OUTPUT_DIR = Path(
    "outputs/t3_ml_canonical"
)

print(
    f"REPO_ROOT={ROOT}"
)
print(
    f"INPUT={INPUT_PATH}"
)
print(
    "VERSIONED_CANONICAL_OUTPUT="
    f"{VERSIONED_OUTPUT_DIR}"
)
print(
    "ARTIFACT_CONTRACT="
    f"{len(ARTIFACT_FILENAMES)}"
)


## Regeneração isolada

O notebook não sobrescreve os artefatos versionados. O runner é executado em diretório temporário e seus resultados são comparados byte a byte aos outputs canônicos presentes no repositório.


In [ ]:
temp_run = tempfile.TemporaryDirectory(
    prefix="case04-notebook03-"
)

TEMP_OUTPUT_DIR = (
    Path(
        temp_run.name
    )
    / "t3_ml_canonical"
)

run_result = run_pipeline(
    input_path=INPUT_PATH,
    output_dir=TEMP_OUTPUT_DIR,
)

print(
    json.dumps(
        {
            key: value
            for key, value
            in run_result.items()
            if key != "output_dir"
        },
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)

splits = pd.read_csv(
    TEMP_OUTPUT_DIR
    / "02_split_distribution.csv"
)

manifest = json.loads(
    (
        TEMP_OUTPUT_DIR
        / "08_run_manifest.json"
    ).read_text(
        encoding="utf-8"
    )
)

print(
    splits.to_string(
        index=False
    )
)


In [ ]:
metrics = pd.read_csv(
    TEMP_OUTPUT_DIR
    / "03_metrics_summary.csv"
)

thresholds = pd.read_csv(
    TEMP_OUTPUT_DIR
    / "04_threshold_metrics_calibration.csv"
)

selected = thresholds.loc[
    thresholds[
        "selected_statistical_baseline"
    ].astype(bool)
]

if len(selected) != 1:
    raise RuntimeError(
        "Threshold estatístico deveria ser único."
    )

selected_threshold = float(
    selected[
        "threshold"
    ].iloc[0]
)

if abs(
    selected_threshold
    - 0.3
) > 1e-12:
    raise RuntimeError(
        "Threshold estatístico divergente."
    )

if not thresholds[
    "operationally_homologated"
].eq(
    False
).all():
    raise RuntimeError(
        "Threshold foi homologado indevidamente."
    )

print(
    "SELECTED_THRESHOLD="
    f"{selected_threshold:.1f}"
)
print(
    "SELECTION_RULE="
    f"{selected['selection_rule'].iloc[0]}"
)
print(
    "OPERATIONALLY_HOMOLOGATED=False"
)

print(
    metrics.to_string(
        index=False
    )
)

print(
    thresholds.to_string(
        index=False
    )
)


In [ ]:
importance = pd.read_csv(
    TEMP_OUTPUT_DIR
    / "05_feature_importance_gain.csv"
)

shap_summary = pd.read_csv(
    TEMP_OUTPUT_DIR
    / "06_shap_summary_test.csv"
)

if len(importance) != 66:
    raise RuntimeError(
        "Feature importance deveria conter "
        "66 features transformadas."
    )

if len(shap_summary) != 66:
    raise RuntimeError(
        "SHAP deveria conter "
        "66 features transformadas."
    )

if not shap_summary[
    "split"
].eq(
    "test"
).all():
    raise RuntimeError(
        "SHAP deve usar somente o teste temporal."
    )

if not shap_summary[
    "rows_explained"
].eq(
    2498
).all():
    raise RuntimeError(
        "SHAP deve explicar os "
        "2.498 registros de teste."
    )

print(
    "FEATURE_IMPORTANCE_TYPE=gain"
)
print(
    "SHAP_SPLIT=test"
)
print(
    "SHAP_TEST_ROWS=2498"
)
print(
    "SHAP_USED_FOR_THRESHOLD_SELECTION=False"
)

print(
    importance.head(
        10
    ).to_string(
        index=False
    )
)

print(
    shap_summary.head(
        10
    ).to_string(
        index=False
    )
)


In [ ]:
versioned_hashes = {
    name: sha256_file(
        VERSIONED_OUTPUT_DIR
        / name
    )
    for name in ARTIFACT_FILENAMES
}

regenerated_hashes = {
    name: sha256_file(
        TEMP_OUTPUT_DIR
        / name
    )
    for name in ARTIFACT_FILENAMES
}

hash_comparison = pd.DataFrame(
    [
        {
            "artifact": name,
            "versioned_sha256":
                versioned_hashes[
                    name
                ],
            "regenerated_sha256":
                regenerated_hashes[
                    name
                ],
            "match":
                versioned_hashes[
                    name
                ]
                == regenerated_hashes[
                    name
                ],
        }
        for name in ARTIFACT_FILENAMES
    ]
)

print(
    hash_comparison.to_string(
        index=False
    )
)

if not hash_comparison[
    "match"
].all():
    raise RuntimeError(
        "Os outputs regenerados não coincidem "
        "byte a byte com os artefatos versionados."
    )

if manifest[
    "data"
][
    "synthetic"
] is not True:
    raise RuntimeError(
        "Manifest deve registrar base sintética."
    )

if manifest[
    "data"
][
    "weak_label"
] is not True:
    raise RuntimeError(
        "Manifest deve registrar label fraco."
    )

if manifest[
    "data"
][
    "r17_in_label"
] is not False:
    raise RuntimeError(
        "R17 não deve integrar o label canônico."
    )

if manifest[
    "threshold"
][
    "operationally_homologated"
] is not False:
    raise RuntimeError(
        "Threshold não pode estar homologado."
    )

if manifest[
    "governance"
][
    "human_review_required"
] is not True:
    raise RuntimeError(
        "Revisão humana deve ser obrigatória."
    )

if manifest[
    "governance"
][
    "production_validation_claimed"
] is not False:
    raise RuntimeError(
        "Não deve existir alegação "
        "de validação produtiva."
    )

print(
    "BYTE_FOR_BYTE_REPRODUCIBILITY=PASS"
)
print(
    "SYNTHETIC_DATA=True"
)
print(
    "WEAK_LABEL=True"
)
print(
    "R17_IN_CANONICAL_LABEL=False"
)
print(
    "HUMAN_REVIEW_REQUIRED=True"
)
print(
    "PRODUCTION_VALIDATION_CLAIMED=False"
)


In [ ]:
temp_path = Path(
    temp_run.name
)

temp_run.cleanup()

print(
    "NOTEBOOK_TEMP_CLEANUP="
    f"{not temp_path.exists()}"
)


## Limitações e interpretação

O resultado deve ser lido como **experimento reproduzível sobre um proxy**, e não como validação independente de atividade suspeita.

Há circularidade conceitual porque o `weak_label` deriva das regras M01–M12 e algumas features podem representar conceitos correlatos. O split é temporal, porém as mesmas entidades aparecem em meses sucessivos; portanto, não há independência completa entre clientes de treino, calibragem e teste.

O threshold `0.3` é somente o baseline estatístico que maximiza MCC na calibragem. Antes de qualquer uso operacional seria necessário incorporar capacidade real da fila, custo de falsos positivos e falsos negativos, objetivo de recall, calibragem, drift, generalização e homologação humana.

A explicabilidade SHAP utiliza somente setembro/2025 após o modelo e o threshold já estarem definidos. Nenhuma explicação é usada para ajustar o treinamento ou selecionar o threshold.

O modelo não é persistido por este notebook e nenhuma alegação de API, deploy, inferência produtiva ou decisão autônoma é feita.
